# Malaika Village Clinic Server (Tier 1) — Fine-Tuned Gemma 4 E4B

**Run on Kaggle T4 (preferred) or Colab T4. Free tier works.**

This notebook is the *receipt* for §10 of `VIDEO_SCRIPT.md` — the "Two-Tier Vision" claim that Malaika scales from offline-phone (Tier 0) to a village clinic with basic internet (Tier 1).

## Two-Tier Architecture

```
TIER 0 — REMOTEST VILLAGE (no internet)
+--------------------------------------------------+
|  $60 Android phone                               |
|  Gemma 4 E2B (2.58 GB, fully offline)            |
|  - Voice in/out (Android native CPU)             |
|  - Vision: alertness, eyes, ribs, edema          |
|  - 12-skill agentic IMCI assessment              |
|  - Deterministic WHO classification              |
+--------------------------------------------------+
                  |
                  | (intermittent Wi-Fi to clinic)
                  v
TIER 1 — VILLAGE CLINIC (basic internet — 10-20 km)
+--------------------------------------------------+
|  Single $300 mini-PC or NUC, on the clinic LAN   |
|  Gemma 4 E4B (4.5B params)                       |
|  + LoRA adapter (ICBHI 2017, merged)             |
|  - Breath-sound spectrogram classification       |
|  - Multi-image vision comparison                 |
|  - Deeper multilingual reasoning                 |
+--------------------------------------------------+
```

The phone always works. The clinic server augments. Either tier is independently useful.

## What This Notebook Proves

1. We can load `Vimal0703/malaika-breath-sounds-E4B-merged` (the merged fine-tuned model from notebook 06) on a single T4 GPU.
2. We can wrap it in a FastAPI server exposing `POST /breath` (audio file in, JSON classification out).
3. We can publish the URL via ngrok so a phone outside the Colab/Kaggle network can call it — simulating phone-in-village calling clinic-on-Wi-Fi.
4. End-to-end latency, throughput, and accuracy numbers — measured live on held-out ICBHI 2017 patients.

See: `docs/NOTEBOOK_12_VILLAGE_CLINIC_PLAN.md` and `VIDEO_SCRIPT.md` §9-10.

## Setup Checklist

| Resource | Where | Required |
|----------|-------|----------|
| `HF_TOKEN` | Kaggle Secrets / Colab userdata | Yes — to load merged model |
| `NGROK_TOKEN` | Kaggle Secrets / Colab userdata | Optional — public URL |
| ICBHI 2017 dataset | `vbookshelf/respiratory-sound-database` via Add Data | Yes — for held-out test |
| GPU | Kaggle T4 / Colab T4 | Yes |

In [ ]:
%%capture
# Cell 1: Install dependencies
!pip install unsloth fastapi pyngrok uvicorn[standard] librosa soundfile Pillow python-multipart

In [ ]:
# Cell 2: Authenticate (Kaggle Secrets first, Colab userdata fallback)
import os
from huggingface_hub import login

HF_TOKEN = None
NGROK_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("Auth source: Kaggle Secrets")
    try:
        NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
    except Exception:
        print("NGROK_TOKEN not set in Kaggle Secrets (ngrok will be skipped)")
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("Auth source: Colab userdata")
        try:
            NGROK_TOKEN = userdata.get("NGROK_TOKEN")
        except Exception:
            print("NGROK_TOKEN not set in Colab userdata (ngrok will be skipped)")
    except Exception:
        print("Neither Kaggle Secrets nor Colab userdata available — manual login")
        login()

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face: authenticated")

print(f"NGROK_TOKEN present: {bool(NGROK_TOKEN)}")

In [ ]:
# Cell 3: Load fine-tuned merged model (Vimal0703/malaika-breath-sounds-E4B-merged)
import os
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

from unsloth import FastModel
import torch

# Skip Unsloth telemetry — same defensive pattern as notebook 06
import unsloth.models._utils as _u
import unsloth.models.vision as _v
_noop = lambda *a, **kw: None
_u.get_statistics = _noop
_v.get_statistics = _noop

model, tokenizer = FastModel.from_pretrained(
    model_name="Vimal0703/malaika-breath-sounds-E4B-merged",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)

print(f"Model loaded: Vimal0703/malaika-breath-sounds-E4B-merged")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Cell 4: Audio -> spectrogram pipeline + ICBHI parser
# Copied verbatim from notebook 06 to guarantee identical training/inference distribution.
import json, random, re, time
from collections import Counter
from pathlib import Path
import glob as _glob

import numpy as np
import librosa
from PIL import Image

SPEC_W, SPEC_H = 512, 256
SPEC_DIR = Path("/kaggle/working/spectrograms")
SPEC_DIR.mkdir(parents=True, exist_ok=True)

def audio_to_spec(audio_path, output_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        if len(y) == 0: return False
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=50, fmax=4000)
        db = librosa.power_to_db(mel, ref=np.max)
        mn, mx = db.min(), db.max()
        norm = ((db - mn) / (mx - mn) * 255).astype(np.uint8) if mx > mn else np.zeros_like(db, dtype=np.uint8)
        norm = np.flip(norm, axis=0)
        Image.fromarray(norm, mode="L").resize((SPEC_W, SPEC_H), Image.Resampling.LANCZOS).convert("RGB").save(str(output_path))
        return True
    except Exception as e:
        print(f"  Skip {Path(audio_path).name}: {e}")
        return False

def parse_icbhi_annotations(audio_dir):
    records = []
    for txt_file in sorted(audio_dir.glob("*.txt")):
        wav_file = txt_file.with_suffix(".wav")
        if not wav_file.exists(): continue
        parts = txt_file.stem.split("_")
        patient_id = parts[0] if parts else "unknown"
        has_crackle, has_wheeze = False, False
        with open(txt_file) as f:
            for line in f:
                fields = line.strip().split()
                if len(fields) >= 4:
                    if int(fields[2]) == 1: has_crackle = True
                    if int(fields[3]) == 1: has_wheeze = True
        if has_crackle and has_wheeze: label = "both"
        elif has_crackle: label = "crackle"
        elif has_wheeze: label = "wheeze"
        else: label = "normal"
        records.append({"audio_path": str(wav_file), "patient_id": patient_id,
            "label": label, "has_crackle": has_crackle, "has_wheeze": has_wheeze})
    return records

def extract_json(text):
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'```\s*$', '', cleaned)
    m = re.search(r'\{[^{}]*\}', cleaned)
    if m:
        return json.loads(m.group(0))
    return None

# Locate ICBHI dataset (auto-detect Kaggle path, same as notebook 06)
_candidates = [
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database",
]
ICBHI_PATH = None
for _c in _candidates:
    if Path(_c).exists():
        ICBHI_PATH = Path(_c)
        break
if ICBHI_PATH is None:
    _found = _glob.glob("/kaggle/input/**/audio_and_txt_files", recursive=True)
    if _found:
        ICBHI_PATH = Path(_found[0]).parent

audio_dir = ICBHI_PATH / "audio_and_txt_files" if ICBHI_PATH else None
audio_files = list(audio_dir.glob("*.wav")) if audio_dir and audio_dir.exists() else []

if audio_files:
    print(f"ICBHI dataset found at: {ICBHI_PATH}")
    print(f"Audio files: {len(audio_files)}")
    records = parse_icbhi_annotations(audio_dir)
    print(f"Parsed {len(records)} recordings (4-class):")
    for label, count in sorted(Counter(r["label"] for r in records).items()):
        print(f"  {label}: {count}")

    # Patient-level split — IDENTICAL seed and ratios to notebook 06
    pids = list(set(r["patient_id"] for r in records))
    random.seed(42); random.shuffle(pids)
    train_pats = set(pids[:int(len(pids)*0.8)])

    INSTRUCTION = ("This is a mel-spectrogram of a child's breathing audio.\n"
        "Vertical: frequency (50-4000 Hz). Horizontal: time. Brightness: intensity.\n"
        "Are the breath sounds normal or abnormal?\n"
        'Respond with JSON: {"abnormal": true/false, "confidence": 0.0-1.0, "description": "brief reason"}')

    test_pairs = [
        {"audio_path": r["audio_path"], "patient_id": r["patient_id"],
         "label": "abnormal" if r["label"] != "normal" else "normal",
         "original_label": r["label"], "instruction": INSTRUCTION}
        for r in records if r["patient_id"] not in train_pats
    ]
    train_pairs = [
        {"audio_path": r["audio_path"], "patient_id": r["patient_id"],
         "label": "abnormal" if r["label"] != "normal" else "normal",
         "original_label": r["label"]}
        for r in records if r["patient_id"] in train_pats
    ]

    print(f"\nPatient-level split (seed=42, 80/20):")
    print(f"  Train patients: {len(train_pats)} -> {len(train_pairs)} recordings")
    print(f"  Held-out patients: {len(pids) - len(train_pats)} -> {len(test_pairs)} recordings")
    print(f"  Held-out distribution: {dict(Counter(p['label'] for p in test_pairs))}")
else:
    print("ERROR: ICBHI dataset not found.")
    print("Add 'respiratory-sound-database' by vbookshelf via 'Add Data' on Kaggle.")
    train_pairs, test_pairs = [], []
    INSTRUCTION = ("This is a mel-spectrogram of a child's breathing audio.\n"
        "Vertical: frequency (50-4000 Hz). Horizontal: time. Brightness: intensity.\n"
        "Are the breath sounds normal or abnormal?\n"
        'Respond with JSON: {"abnormal": true/false, "confidence": 0.0-1.0, "description": "brief reason"}')

In [ ]:
# Cell 5: Inference helpers — classify_breath_sound + generate_clinical_note
from transformers import AutoProcessor
from PIL import Image
import io

processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")
print("Processor loaded")

def classify_breath_sound(wav_bytes: bytes) -> dict:
    """Take raw WAV bytes, return classification dict.

    Pipeline: bytes -> /tmp WAV -> mel-spectrogram PNG -> Gemma 4 E4B (merged) -> JSON.
    Identical preprocessing path to notebook 06 training distribution.
    """
    wav_path = "/tmp/incoming.wav"
    spec_path = "/tmp/incoming_spec.png"
    with open(wav_path, "wb") as f:
        f.write(wav_bytes)
    if not audio_to_spec(wav_path, spec_path):
        return {"error": "could not process audio"}
    img = Image.open(spec_path).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": INSTRUCTION},
    ]}]
    txt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=txt, images=[img], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = extract_json(raw)
    if parsed is None:
        return {"error": "parse failed", "raw": raw}
    return parsed

CLINICAL_NOTE_PROMPT = """You are the senior nurse at a rural pediatric clinic, mentoring a junior colleague.

A breath-sound recording from a child has just been analyzed by the auscultation AI.

Auscultation AI result:
- Classification: {classification}
- Confidence: {confidence:.0%}
- Technical description: {description}

Write a 3-4 sentence clinical note for the patient chart, in the voice of an experienced senior nurse. Use precise clinical language. Ground your reasoning in WHO IMCI protocol. End with one specific next-step recommendation.

Output rules:
- The note only. No preamble. No JSON. No quotes around it. No markdown.
- 3 to 4 sentences total.
- Reference WHO IMCI when the classification dictates next steps (e.g. count respiratory rate, look for chest indrawing, check for general danger signs).

Begin the note now:"""

def generate_clinical_note(classification: dict) -> str:
    """Generate a 3-4 sentence clinical reasoning note via the same Gemma 4 model.

    This is the 'consulting senior nurse' voice — turns a binary classification
    into the kind of chart entry an experienced clinician would write.
    """
    cls_label = "ABNORMAL breath sounds" if classification.get("abnormal") else "NORMAL breath sounds"
    confidence = classification.get("confidence") or 0.0
    description = classification.get("description") or "(no technical description)"
    prompt = CLINICAL_NOTE_PROMPT.format(
        classification=cls_label,
        confidence=confidence,
        description=description,
    )
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    txt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=txt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.05,
        )
    note = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    note = note.strip()
    # Strip common preamble / wrapping the model occasionally produces
    for prefix in ('"', "'", "Note:", "Clinical note:", "Chart entry:"):
        if note.lower().startswith(prefix.lower()):
            note = note[len(prefix):].strip()
    if note.endswith(('"', "'")):
        note = note[:-1].strip()
    return note

print("classify_breath_sound() ready")
print("generate_clinical_note() ready")

In [ ]:
# Cell 6: FastAPI server with /health and /breath endpoints
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn, threading, time

app = FastAPI(title="Malaika Village Clinic Server")

@app.get("/health")
def health():
    return {"status": "ok", "model": "gemma-4-E4B-merged-breathsound"}

@app.post("/breath")
async def breath(audio: UploadFile = File(...)):
    if not audio.filename or not audio.filename.lower().endswith((".wav", ".mp3", ".m4a")):
        return JSONResponse({"error": "audio file required (.wav/.mp3/.m4a)"}, status_code=400)
    wav_bytes = await audio.read()
    result = classify_breath_sound(wav_bytes)
    # Add the clinical reasoning note — same Gemma 4 model, second pass.
    if "error" not in result:
        try:
            result["clinical_note"] = generate_clinical_note(result)
        except Exception as e:
            result["clinical_note_error"] = f"{type(e).__name__}: {e}"
    return JSONResponse(result)

PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print(f"FastAPI server running on :{PORT}")
print(f"  GET  /health")
print(f"  POST /breath  (multipart audio file)")

In [ ]:
# Cell 7: Expose via ngrok (so a phone outside the Kaggle/Colab network can call us)
public_url = None

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    if public_url.startswith("http://"):
        public_url = public_url.replace("http://", "https://", 1)

    print("=" * 60)
    print("  MALAIKA VILLAGE CLINIC SERVER IS LIVE")
    print("=" * 60)
    print(f"\n  Public URL: {public_url}")
    print(f"\n  Health check: {public_url}/health")
    print(f"  Endpoint:     POST {public_url}/breath")
    print(f"\n  curl example:")
    print(f"    curl -X POST -F 'audio=@sample.wav' {public_url}/breath")
    print("\n" + "=" * 60)
else:
    print("NGROK_TOKEN not set — public URL skipped.")
    print(f"Server is reachable inside this VM at http://localhost:{PORT}")
    print("To get a public URL, add NGROK_TOKEN to Kaggle Secrets / Colab userdata and re-run.")
    public_url = f"http://localhost:{PORT}"

In [ ]:
# Cell 8: Live demo — pick a held-out abnormal sample and call the endpoint
import requests

if test_pairs:
    abnormal_pool = [p for p in test_pairs if p["label"] == "abnormal"]
    if not abnormal_pool:
        print("No abnormal held-out samples found — falling back to any test sample.")
        abnormal_pool = test_pairs

    random.seed(42)
    demo_pair = random.choice(abnormal_pool)

    print("=" * 60)
    print("LIVE DEMO — phone calling clinic server (held-out patient)")
    print("=" * 60)
    print(f"Patient ID:      {demo_pair['patient_id']} (NOT in training set)")
    print(f"True label:      {demo_pair['label']} (original: {demo_pair['original_label']})")
    print(f"Audio file:      {Path(demo_pair['audio_path']).name}")
    print(f"Endpoint:        {public_url}/breath")
    print()

    t0 = time.monotonic()
    with open(demo_pair["audio_path"], "rb") as f:
        response = requests.post(
            f"{public_url}/breath",
            files={"audio": (Path(demo_pair["audio_path"]).name, f, "audio/wav")},
            timeout=60,
        )
    elapsed = time.monotonic() - t0

    print(f"Status:          {response.status_code}")
    print(f"Round-trip time: {elapsed:.2f}s")
    print(f"Response:")
    print(json.dumps(response.json(), indent=2))
else:
    print("No held-out test data — cannot run live demo.")
    print("Confirm the ICBHI dataset is attached to this Kaggle notebook.")

In [ ]:
# Cell 9: Throughput benchmark — 20 held-out samples, measure latency + accuracy
if test_pairs:
    print("=" * 60)
    print("THROUGHPUT BENCHMARK — 20 held-out samples")
    print("=" * 60)

    sample = test_pairs[:20]
    correct = 0
    errors = 0
    latencies = []

    t_total = time.monotonic()
    for i, pair in enumerate(sample):
        t0 = time.monotonic()
        try:
            with open(pair["audio_path"], "rb") as f:
                r = requests.post(
                    f"{public_url}/breath",
                    files={"audio": (Path(pair["audio_path"]).name, f, "audio/wav")},
                    timeout=60,
                ).json()
        except Exception as e:
            print(f"  [{i+1:2d}/20] ERROR: {e}")
            errors += 1
            continue
        dt = time.monotonic() - t0
        latencies.append(dt)

        is_abn = r.get("abnormal")
        expected = pair["label"] == "abnormal"
        ok = (is_abn == expected)
        if ok: correct += 1
        marker = "PASS" if ok else "FAIL"
        print(f"  [{i+1:2d}/20] {marker} expected={pair['label']:8s} got={str(is_abn):5s} ({dt:.2f}s)")

    elapsed = time.monotonic() - t_total
    n = len(latencies)
    print()
    print(f"Accuracy:    {correct}/{len(sample)} ({100*correct/len(sample):.0f}%)")
    print(f"Errors:      {errors}")
    if n:
        print(f"Latency:     mean={sum(latencies)/n:.2f}s  min={min(latencies):.2f}s  max={max(latencies):.2f}s")
    print(f"Total time:  {elapsed:.1f}s")
    print(f"Throughput:  {len(sample)*60/elapsed:.1f} requests/min")
else:
    print("No held-out test data — cannot run benchmark.")

In [ ]:
# Cell 10: Privacy architecture + how to integrate with the phone
print("=" * 60)
print("PRIVACY ARCHITECTURE")
print("=" * 60)
print("""
This server runs entirely in the clinic's own infrastructure.
The audio file is processed on the GPU and discarded.
No data is logged. No data is persisted. No data is sent anywhere else.

In production (clinic deployment):
  - Replace ngrok with the clinic's local IP on the LAN.
  - The phone calls the server only when on the clinic Wi-Fi.
  - When off the clinic Wi-Fi, the phone falls back to Gemma 4 E2B
    on-device — no audio analysis, but full IMCI assessment still works.

Integration in malaika_flutter:
  - Setting -> "Connected mode -> Clinic server URL"
  - When detected on clinic Wi-Fi (SSID match) -> enable /breath endpoint calls.
  - Otherwise -> on-device E2B only.

This is what 'two-tier deployment' actually means in code.
""")

In [ ]:
# Cell 11: Summary card
print("=" * 60)
print("VILLAGE CLINIC SERVER SUMMARY")
print("=" * 60)
print(f"Base model:        google/gemma-4-E4B-it (4.5B params)")
print(f"Adapter:           Unsloth QLoRA, r=8, ICBHI 2017 (merged)")
print(f"Hosted model:      Vimal0703/malaika-breath-sounds-E4B-merged")
_train_n = len(train_pairs) if 'train_pairs' in dir() else 0
_test_n = len(test_pairs) if 'test_pairs' in dir() else 0
print(f"Trained on:        {_train_n} balanced spectrograms (notebook 06)")
print(f"Held-out patients: {_test_n} recordings (this notebook)")
print(f"Held-out accuracy: 85% crackle, 40% overall (binary, hackathon-grade)")
print(f"Endpoint:          POST /breath  (audio file -> JSON classification)")
print(f"Privacy:           clinic-local server, no cloud, no logs")
print(f"GPU:               {torch.cuda.get_device_name(0)}")
print(f"VRAM:              {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Latency (T4):      ~3-5s per audio sample")
print()
print("This is Tier 1 of Malaika's two-tier deployment.")
print("Tier 0 (phone, fully offline) -- see malaika_flutter/")
print("Tier 1 (clinic, basic internet) -- this notebook")
print("=" * 60)